# qAIR-vNext **v44** -- Local Training (cell by cell)

Runs the same steps as `python main.py --mode train`
(`training/runner.py::run_training`), split into separate cells so each
stage can be inspected before moving to the next.

> **v44 is the post-audit release.** A full technical audit found that the
> quantum circuit, the multi-hypothesis superposition and the Born-rule
> collapse were **all provably inert** -- the circuit was bit-exactly
> independent of its input, the reasoner collapsed every hypothesis to one
> constant vector, and the model's prediction was unchanged when `H` was
> replaced with zeros. **Every quantum and ablation number produced before
> v44 is void.** See the README's audit section for the measurements.
>
> Two consequences for this notebook:
> * `QAIRvNext.forward` now takes `Q` (question embedding) and the padding
>   masks. Older notebooks (`qAIR_v43`, `qair_v4[02]_colab`) will raise.
> * Section 20 runs the **input-ablation test**, which is the only cell here
>   that can tell you whether the model is actually using its hypotheses.
>   Accuracy cannot. Do not report results without it.

This is for **local** use (your PC or Mac), not Colab -- no Drive mount, no
repo clone. It assumes you've already run, in this repo's root:

```
python -m venv venv
.\venv\Scripts\Activate.ps1      # or: source venv/bin/activate
pip install -r requirements.txt
```

## 0. Make sure we're running from the repo root

This notebook lives in `notebooks/`, but every `from models...` /
`from training...` import and every relative path in `config.py`
(`./cache`, `./ckpt`) assumes the working directory is the repo root.
Jupyter/VS Code often default the kernel's cwd to the notebook's own
folder -- fix it here if so, once, before anything else runs.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working directory:", os.getcwd())
assert os.path.exists("config.py"), (
    "Not at the repo root -- adjust the os.chdir(...) above to point "
    "here manually, e.g. os.chdir(r'D:\\WORK\\Research\\qAIR-CSE499B')"
)

Working directory: d:\WORK\Research\qAIR-CSE499B


## 0b. Force offline mode for Hugging Face downloads

The Qwen2.5-0.5B-Instruct model and the MiniLM encoder are already
fully downloaded and cached locally (verified: ~953MB in
`~/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct`).
Without this, `huggingface_hub` still makes a network round-trip on
every load to check whether the cached files are still current --
and on this machine that request appears to hang indefinitely
(confirmed: the kernel process sits at 0% CPU, i.e. blocked on I/O,
not computing) rather than failing fast, likely a firewall/proxy/AV
silently swallowing the request.

Setting `HF_HUB_OFFLINE=1` skips that network check entirely and
loads straight from the local cache. **If you ever need to download a
*new* model for the first time on this machine** (or run this
notebook somewhere nothing is cached yet, e.g. a fresh Mac), comment
this cell out first -- offline mode will fail fast with a clear "file
not found" error instead of downloading, which is what you want.

In [2]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"

print("HF_HUB_OFFLINE:", os.environ["HF_HUB_OFFLINE"])

HF_HUB_OFFLINE: 1


## 1. Imports, seed, device

`resolve_device()` picks cuda > mps > cpu -- confirm here which one
your machine actually got before committing to a full run.

In [3]:
from functools import partial

import torch
from torch.utils.data import DataLoader

from config import (
    CACHE_DIR,
    CKPT_DIR,
    EPOCHS,
    PATIENCE,
    PERSISTENT_STEPS,
    N_QUBITS,
    BATCH_SIZE,
    WEIGHT_DECAY,
    SEED,
    EMBEDDING_DIM as DIM,
    resolve_device,
)

from training.seed import set_seed, seeded_generator
from training.dataset import QAIRDataset, collate_fn
from training.train import Trainer
from training.checkpoint import load_or_resume
from training.evaluate import evaluate
from models.full_model import QAIRvNext

# set_seed existed for the project's whole history and was never called
# from any entry point, so no run was reproducible. It is called from
# main.py / runner.py / ablations.py now, and here.
set_seed(SEED)

device = resolve_device()
print("Device:", device)

d:\WORK\Research\qAIR-CSE499B\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SEED] Set to 42 (deterministic=False)
Device: cpu


## 2. Configuration for this run

Same parameters `run_training()` takes. Set `TRAIN_SAMPLES`/`VAL_SAMPLES`
to a small number (e.g. 40/20) for a fast smoke test of the whole
pipeline before committing to the full dataset -- `None` means "use
everything".

In [4]:
RUN_NAME = "qair_v44"

TRAIN_SAMPLES = None   # e.g. 40 for a quick smoke test
VAL_SAMPLES = None     # e.g. 20 for a quick smoke test

epochs = EPOCHS
persistent_steps = PERSISTENT_STEPS   # 3 in v44, down from 5 -- NOT re-tuned since the collapse fix
n_qubits = N_QUBITS

print(f"epochs={epochs}  persistent_steps={persistent_steps}  n_qubits={n_qubits}")
print(f"batch_size={BATCH_SIZE}  weight_decay={WEIGHT_DECAY}  patience={PATIENCE}  seed={SEED}")

epochs=30  persistent_steps=3  n_qubits=6
batch_size=16  weight_decay=0.02  patience=5  seed=42


## 3. Build (or load cached) datasets

First run builds `cache/arc_*.pt` from scratch -- one hypothesis per
answer option via the LLM for every ARC question, then embeds question,
hypotheses and options. Slow, one-time; every later run reuses it.

Two things happen automatically at load:

* **Auto-migration.** A pre-v33 cache gains its **question embeddings**
  in place (encoder only, no LLM -- about a minute). The question text
  was always cached; its embedding never was, and adding it is worth
  **+12.6 points** on a probe (0.3585 -> 0.4849).
* **Quality report.** The fallback rate is printed. Migration **cannot**
  repair hypothesis quality -- if you see a `[CACHE QUALITY WARNING]`,
  the cached hypotheses are template noise carrying no information about
  which option is correct. **Delete `cache/arc_*.pt` and rebuild.** The
  shipped cache was 33% fallbacks with broken hypothesis/option
  alignment; that is the single biggest thing still limiting this model.

In [ ]:
train_ds = QAIRDataset(split="train", max_samples=TRAIN_SAMPLES, cache_dir=CACHE_DIR)
val_ds = QAIRDataset(split="validation", max_samples=VAL_SAMPLES, cache_dir=CACHE_DIR)

print(f"Train samples: {len(train_ds)}")
print(f"Val samples:   {len(val_ds)}")

[CACHE MISSING] Building train cache from scratch...


Using the latest cached version of the dataset since ai2_arc couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'ARC-Challenge' at C:\Users\User\.cache\huggingface\datasets\ai2_arc\ARC-Challenge\0.0.0\210d026faf9955653af8916fad021475a3f00453 (last modified on Sat Jul 25 01:07:17 2026).


[ENCODER] Using sentence-transformers/all-MiniLM-L6-v2
[ENCODER] Embedding dimension: 384


Using the latest cached version of the dataset since ai2_arc couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'ARC-Easy' at C:\Users\User\.cache\huggingface\datasets\ai2_arc\ARC-Easy\0.0.0\210d026faf9955653af8916fad021475a3f00453 (last modified on Sat Jul 25 01:07:34 2026).
Building train cache:   2%|▏         | 56/3370 [06:26<6:10:52,  6.71s/it]

[AUTOSAVE] 50 samples (raw_index=50)


...

Building train cache:  25%|██▌       | 856/3370 [1:35:17<4:18:43,  6.17s/it]

[AUTOSAVE] 850 samples (raw_index=850)


Building train cache:  27%|██▋       | 904/3370 [1:40:07<4:08:06,  6.04s/it]

[AUTOSAVE] 900 samples (raw_index=900)


Building train cache:  27%|██▋       | 912/3370 [1:41:01<4:15:08,  6.23s/it]

## 3b. Inspect a single sample

Before training on it -- confirm the hypotheses look reasonable and
the embedding shapes match `config.EMBEDDING_DIM`.

In [ ]:
sample = train_ds[0]

print("Question:", sample["question"])
print()
print("Options:", sample["options"])
print()
print("Generated hypotheses (* = fallback, carries NO signal):")
flags = sample.get("is_fallback", [False] * len(sample["hypotheses"]))
for i, h in enumerate(sample["hypotheses"]):
    mark = "*" if bool(flags[i]) else " "
    print(f" {mark}{i}. {h}")
print()
print("Q shape:", sample["Q"].shape if "Q" in sample else "MISSING -- rebuild cache")
print("H shape:", sample["H"].shape)
print("O shape:", sample["O"].shape)
print("Correct answer index (y):", sample["y"])

## 4. DataLoaders

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=seeded_generator(SEED),   # batch ORDER reproducible too, not just weight init
    collate_fn=partial(collate_fn, shuffle_options=True),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=partial(collate_fn, shuffle_options=False),
)

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 4b. Inspect a single batch

In [ ]:
batch = next(iter(train_loader))

for k, v in batch.items():
    print(f"{k:8s} shape={tuple(v.shape)} dtype={v.dtype}")

## 5. Build the model

`use_quantum=True, use_validator=True` matches `run_training()`'s defaults.

`use_question=True` (v44) threads the question embedding through the
validator and selector. `validator_feedback=True` closes the loop that was
open for the project's whole history: the validator's `potential` field was
computed, weighted, returned -- and then silently dropped, because
`forward` called `self.reasoner(H)` without it.

Note the feedback pass runs the reasoner **and the quantum layer twice** per
forward. Since the CPU-bound circuit dominates, set
`validator_feedback=False` to roughly halve epoch time while iterating.

In [ ]:
model = QAIRvNext(
    dim=DIM,
    use_quantum=True,
    use_validator=True,
    persistent_steps=persistent_steps,
    n_qubits=n_qubits,
    use_question=True,
    validator_feedback=True,
    verbose=True,           # prints the quantum/classical layer parameter counts
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")
print(f"Model device: {next(model.parameters()).device}")

## 6. Trainer + checkpoint resume

If `ckpt/{RUN_NAME}_latest.pt` already exists (from a previous run of
this notebook), this resumes from it instead of starting over.

In [ ]:
import os

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    ckpt_dir=CKPT_DIR,
    name=RUN_NAME,
    weight_decay=WEIGHT_DECAY,
)

latest_ckpt = os.path.join(CKPT_DIR, f"{RUN_NAME}_latest.pt")
print("Checkpoint path:", latest_ckpt)
print("Exists:", os.path.exists(latest_ckpt))

start_epoch, best_acc, _ = load_or_resume(trainer, CKPT_DIR, RUN_NAME, epochs)
print(f"Starting from epoch {start_epoch} / {epochs}  (best_acc so far: {best_acc:.4f})")

## 7. Train

Long-running cell. Interrupting is safe -- a `_latest.pt` checkpoint is
saved every epoch, so re-running (after re-running cell 6's `Trainer`
build) resumes from the last completed epoch.

**Watch `Pairwise Cos` and `H Cos`, not just accuracy.** They measure
whether the hypotheses are still distinguishable after reasoning. Near 1.0
means they have merged into one vector, the collapse distribution is uniform
by construction, and the multi-hypothesis mechanism is inert -- which is
exactly what happened pre-v44 while accuracy read a plausible 33.7%. The
trainer prints a `[COLLAPSE WARNING]` above 0.99 and tells you which of the
two failure modes it is (reasoner merged them vs. selector ignoring them).

In [ ]:
history = trainer.train(
    epochs=epochs,
    start_epoch=start_epoch,
    best_acc=best_acc,
    patience=PATIENCE,
)

## 8. Save the run config alongside the checkpoint

So a later `evaluation/sample_inference.py`-style load can't silently
mismatch `n_qubits`/`persistent_steps` against what this checkpoint
was actually trained with.

In [ ]:
config_path = os.path.join(CKPT_DIR, f"{RUN_NAME}_config.pt")

torch.save(
    {
        "use_quantum": True,
        "use_validator": True,
        "persistent_steps": persistent_steps,
        "n_qubits": n_qubits,
        "use_question": True,
        "seed": SEED,
    },
    config_path,
)

print(f"[CONFIG SAVED] {config_path}")

## 9. Quick validation-set evaluation

Same `evaluate()` used during training, run once more standalone on
the final model state.

In [ ]:
metrics = evaluate(model, val_loader, device)

for k, v in metrics.items():
    print(f"{k:15s}: {v:.4f}")

## 10. Helpers for ablation results

Small local versions of the print/report helpers `qair_v42_colab.ipynb`
uses, scoped to what the sections below need.

In [ ]:
def checkpoint_status(path, label="Checkpoint"):
    print("=" * 60)
    print(label)
    print("=" * 60)
    print("Path:", path)
    print("Exists:", os.path.exists(path))
    if not os.path.exists(path):
        print("[NO CHECKPOINT FOUND] Will train from scratch.")
        return None
    try:
        ckpt = torch.load(path, map_location="cpu")
        epoch = ckpt.get("epoch", "UNKNOWN")
        print(f"epoch = {epoch}")
        if isinstance(epoch, int):
            print(f"start_epoch = {epoch + 1}")
        return ckpt
    except Exception as e:
        print("[CHECKPOINT INCOMPATIBLE / FAILED TO LOAD]")
        print(e)
        return None


def save_report(metrics, export_dir, filename="evaluation.txt"):
    os.makedirs(export_dir, exist_ok=True)
    report_path = os.path.join(export_dir, filename)
    with open(report_path, "w") as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v}\n")
    print("Saved to:", report_path)
    return report_path


def list_exports(export_dir):
    print("=" * 60)
    print("EXPORTED FILES")
    print("=" * 60)
    for f in sorted(os.listdir(export_dir)):
        print(f)


EXPORT_DIR = "./exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

## 11. Check existing ablation checkpoints

All 7 configs in `training/ablations.py::ABLATIONS` -- including the
two parameter-matched classical-control configs (`A1c_...`, `A3c_...`)
added for the quantum-vs-classical comparison. Resume support works
the same way as the single-model training above: any config with a
`_latest.pt` checkpoint picks up where it left off instead of
restarting.

In [ ]:
from training.ablations import run_ablation_suite, ABLATIONS

for name in ABLATIONS.keys():
    print()
    checkpoint_status(os.path.join(CKPT_DIR, f"{name}_latest.pt"), label=name)

## 12. Run the full ablation suite

> ### ⚠️ DO NOT "Run All" through this cell.
> This trains **every config in `ABLATIONS` under every seed in
> `config.SEEDS`** -- 8 x 5 = **40 training runs**. On CPU, with the
> quantum layer running twice per forward (validator feedback), that is
> **days**, not hours.
>
> For a first end-to-end validation, **skip this cell and cell 13**.
> Sections 9, 20 and 21 below work off the model you just trained in
> section 7 and are what actually tell you whether the system works.
>
> When you do run it: start with a subset, e.g.
> `seeds=(42,)` and a couple of configs, and only then commit to the
> full grid -- ideally on a GPU.

Trains/resumes every config under every seed. Single-run numbers are not
interpretable here: ARC's 869-example validation split gives a standard
error of ~1.6 points, larger than most gaps this grid produces.

The load-bearing row is `A1b_quantum_only` vs `A1c_classical_control_only`
-- a parameter-matched classical MLP in the quantum layer's slot, same
total parameter count, so a gap cannot be attributed to capacity.

In [ ]:
# WARNING: 8 configs x 5 seeds = 40 training runs. See the note above.
# Start small -- this is the subset version. Widen once it is cheap.
from training.ablations import run_ablation_suite, ABLATIONS

RUN_FULL_GRID = False   # flip to True only when you can afford it

if RUN_FULL_GRID:
    ablation_results = run_ablation_suite(
        cache_dir=CACHE_DIR,
        ckpt_dir=CKPT_DIR,
        epochs=epochs,
        patience=PATIENCE,
        n_qubits=n_qubits,
        seeds=(SEED,),          # one seed to start
        with_test=False,        # no test cache built yet
    )
    for name, runs in ablation_results.items():
        accs = [r["best_val_acc"] for r in runs]
        print(f"{name:<34s} n={len(runs)}  val={sum(accs)/len(accs):.4f}")
else:
    ablation_results = None
    print("Skipped the ablation grid (RUN_FULL_GRID=False).")
    print("Sections 9, 20 and 21 use the model trained in section 7.")

## 13. Pick the model the evaluation sections below will use

If you ran the ablation grid, load a specific arm's checkpoint here.
If you skipped it, this falls back to the model you trained in section 7,
so sections 14, 20 and 21 work either way.

Checkpoint names carry a seed suffix (`{config}_s{seed}_best.pt`), since
each config is trained once per seed.

In [ ]:
import os

BEST_CONFIG_NAME = "A4_full_hybrid"
EVAL_SEED = SEED

_ckpt = os.path.join(CKPT_DIR, f"{BEST_CONFIG_NAME}_s{EVAL_SEED}_best.pt")

if ablation_results is not None and os.path.exists(_ckpt):
    from evaluation.sample_inference import load_ablation_model
    eval_model = load_ablation_model(
        ckpt_dir=CKPT_DIR,
        name=f"{BEST_CONFIG_NAME}_s{EVAL_SEED}",
        cfg=ABLATIONS[BEST_CONFIG_NAME],
        device=device,
        n_qubits=n_qubits,
    )
    EVAL_LABEL = f"{BEST_CONFIG_NAME}_s{EVAL_SEED}"
else:
    # Reload the best checkpoint from section 7 rather than using the
    # final-epoch weights still sitting in `model`.
    _best = os.path.join(CKPT_DIR, f"{RUN_NAME}_best.pt")
    if os.path.exists(_best):
        model.load_state_dict(torch.load(_best, map_location=device)["model"])
        print(f"[LOADED] best checkpoint {_best}")
    eval_model = model
    EVAL_LABEL = RUN_NAME
    print(f"Using the model trained in section 7 ({EVAL_LABEL}).")

## 14. Full validation-set evaluation

Reuses `val_ds` from section 3. `shuffle_options=False` here (matching
section 9's reasoning) so these metrics are reproducible run to run.

In [ ]:
eval_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=partial(collate_fn, shuffle_options=False),
)

eval_metrics = evaluate(eval_model, eval_loader, device)
for k, v in eval_metrics.items():
    print(f"{k:15s}: {v:.4f}")

report_path = save_report(eval_metrics, EXPORT_DIR, filename=f"{EVAL_LABEL}_evaluation.txt")

## 15. Sample-question sanity check

Runs the 10 hand-written questions in `evaluation/sample_inference.py`
through the loaded model -- a quick, human-readable check independent
of the ARC validation set.

In [ ]:
sample_acc = run_sample_evaluation(model=eval_model, device=device)
print(f"\nSample Accuracy ({EVAL_LABEL}) = {sample_acc:.4f}")

## 16. Grab one sample for visualization

In [ ]:
# keep_trajectory is off by default (it forced a device->host sync on every
# reasoning step of every forward pass, including training). Turn it on just
# for this visualization pass.
eval_model.reasoner.keep_trajectory = True

viz_loader = DataLoader(val_ds, batch_size=1, shuffle=True,
                        collate_fn=partial(collate_fn, shuffle_options=False))
viz_batch = next(iter(viz_loader))

viz_H = viz_batch["H"].to(device)
viz_O = viz_batch["O"].to(device)
viz_Q = viz_batch["Q"].to(device)
viz_Hm = viz_batch["H_mask"].to(device)
viz_Om = viz_batch["O_mask"].to(device)

with torch.no_grad():
    viz_out = eval_model(viz_H, viz_O, Q=viz_Q, H_mask=viz_Hm, O_mask=viz_Om)

eval_model.reasoner.keep_trajectory = False
print("collapse_probs:", viz_out["collapse_probs"][0].cpu().numpy())

## 17. Visualizations

In [ ]:
from visualization.attention_maps import plot_attention_map

attention = viz_out["attention"]
if attention.dim() == 4:
    attention = attention[0, 0]
elif attention.dim() == 3:
    attention = attention[0]

plot_attention_map(attention, save_path=f"{EXPORT_DIR}/{EVAL_LABEL}_attention_map.png")

In [ ]:
from visualization.energy_maps import plot_energy_map
plot_energy_map(viz_out["answer_energy"][0], save_path=f"{EXPORT_DIR}/{EVAL_LABEL}_energy_map.png")

In [ ]:
import matplotlib.pyplot as plt

collapse = viz_out["collapse_probs"][0].detach().cpu().numpy()
plt.figure(figsize=(6, 4))
plt.bar(range(len(collapse)), collapse)
plt.xlabel("Hypothesis")
plt.ylabel("Probability")
plt.title("Collapse Distribution")
plt.savefig(f"{EXPORT_DIR}/{EVAL_LABEL}_collapse_probs.png", bbox_inches="tight")
plt.show()

In [ ]:
from visualization.trajectory import plot_trajectory
plot_trajectory(viz_out["trajectory"], save_path=f"{EXPORT_DIR}/{EVAL_LABEL}_trajectory.png")

In [ ]:
validator_out = viz_out["validator"]
if validator_out is not None:
    scores = [
        validator_out["causal"][0].mean().item(),
        validator_out["diversity"][0].mean().item(),
        validator_out["specificity"][0].mean().item(),
        validator_out["relevance"][0].mean().item(),
    ]
    labels = ["causal", "diversity", "specificity", "relevance"]
    plt.figure(figsize=(7, 4))
    plt.bar(labels, scores)
    plt.title("Validator Scores")
    plt.savefig(f"{EXPORT_DIR}/{EVAL_LABEL}_validator.png", bbox_inches="tight")
    plt.show()
else:
    print(f"{EVAL_LABEL} has use_validator=False -- no validator scores to plot.")

## 18. List exported files

In [ ]:
list_exports(EXPORT_DIR)

## 20. ⭐ Input ablation -- does the model actually use its hypotheses?

**This is the most important cell in the notebook.**

qAIR's claim is that it reasons over multiple hypotheses held in
superposition. That claim is falsifiable in one line: corrupt the
hypotheses and see whether the answer changes. Pre-v44 it did not --
not "changed a little", but **bit-identically unchanged** at 294/869
across every corruption tried (zeros, noise, shuffled, another
question's hypotheses, `H := O`). Only `O` ever mattered.

Accuracy alone could never have revealed that. If this cell reports
FAIL, no number anywhere in this notebook supports a claim about
superposition, collapse, or quantum reasoning -- however good it looks.

In [ ]:
from evaluation.input_ablation import input_ablation, report

results = input_ablation(eval_model, eval_loader, device)
passed = report(results)

print("\nPASSED" if passed else "\nFAILED -- the hypothesis pathway is inert")

## 21. Reference baselines

Two numbers that decide how to read everything above.

**Data ceiling** (`--mode probe`): trainable probes directly on the cached
embeddings. No architecture over this cache can do much better, so a qAIR
number below these is a problem with qAIR, not with the task. Measured
pre-fix on validation:

| probe | val acc |
|---|---|
| O only | 0.3585 |
| diag(H_n, O_n) | 0.3527 |
| full pairwise H_k x O_n | 0.3979 |
| **Q x O pairwise** | **0.4849** |
| Q x O + H | 0.4408 &nbsp; *(the old H made it WORSE)* |

The full 8.2M-parameter pre-fix model scored **0.3383** -- below the
options-only MLP.

**Direct LLM** (`--mode direct`): scores each option's likelihood under
Qwen2.5-0.5B, the same model used to generate hypotheses. If the pipeline
cannot beat the model it is built on top of, it is a lossy compression of
knowledge the generator already had. **This is the first thing a reviewer
will check, and it has not been run yet.**

In [ ]:
from evaluation.baselines import run_probes, run_direct

probe_results = run_probes(CACHE_DIR)

# Needs the LLM -- slower. Use limit=200 for a quick read.
direct_acc, direct_records = run_direct(CACHE_DIR, split="validation", limit=None)

## 22. Statistical comparison between ablation arms

Bootstrap CIs plus **paired** McNemar tests. Paired matters: two arms see
the same examples, so testing the disagreement cells is far more powerful
than comparing two independent proportions.

Reminder on effect sizes: at ~869 examples the standard error is ~1.6
points. A 2-point gap between arms is **not** a result without this cell.

In [ ]:
from evaluation.stats import compare_arms

if ablation_results:
    compare_arms(ablation_results, baseline="A1_baseline")
else:
    print("No ablation results -- run section 12 with RUN_FULL_GRID=True first.")